# Cross-Framework Agent Interoperability via A2A


## What We Are Going To Do

The whole point of the Agent-to-Agent (A2A) protocol is that two agents built with
**completely different internals** can still collaborate, as long as each one exposes
a standard A2A `AgentCard` and speaks the A2A wire protocol. Neither agent needs to know
what framework, prompting style, or tools the other one uses internally.

In this notebook we build exactly that, end to end:

1. **Research Agent** — a `LangGraph` ReAct-style agent (via `from helpers import get_llm`)
   with a lookup/"web search" tool, wrapped as an A2A server with its own `AgentCard`.
2. **Writer Agent** — a deliberately *differently-structured* agent: a single plain LLM
   call with a fixed persona/prompt, no LangGraph, no tools, no agent loop at all — also
   wrapped as its own A2A server with its own `AgentCard`.
3. **A registry-based discovery step** — mirroring `agent_discovery.py`'s pattern of reading
   a JSON list of base URLs and querying each one's `/.well-known/agent.json` — that
   discovers both `AgentCard`s at runtime.
4. **An orchestrating script** that uses an `AgentConnector`-style client wrapper (mirroring
   `agent_connect.py`) to: send a research task to the Research Agent, then feed the
   Research Agent's raw output straight into the Writer Agent as its input, entirely over A2A.
5. Clear "hop" printing at every step, so the cross-agent handoff is fully visible.

This mirrors the pattern already used at larger scale in this repo's
`09_Agent_Protocols/MCP/mcp_a2a_agentic_rag/` applied build (which coordinates an
agentic-RAG agent and a host agent over A2A + MCP together) — here we isolate **just**
the A2A interop mechanic, with no RAG and no MCP, for teaching clarity.


## Discussion of the Approach

We reuse the *exact* conventions this repo already established for real A2A usage in
`09_Agent_Protocols/MCP/mcp_a2a_agentic_rag/utilities/a2a/`:

- `AgentConnector` (from `agent_connect.py`) — a thin wrapper around `a2a.client.A2AClient`
  that exposes a single `send_task(message, session_id)` coroutine.
- `AgentDiscovery` (from `agent_discovery.py`) — reads a registry file (a JSON list of base
  URLs), and for each one calls `A2ACardResolver(...).get_agent_card()` against its
  `/.well-known/agent.json` endpoint.
- Each agent is served with `a2a.server.apps.A2AStarletteApplication` +
  `a2a.server.request_handlers.DefaultRequestHandler` + `a2a.server.tasks.InMemoryTaskStore`,
  exactly like `agents/agentic_rag_agent/main.py` and `agents/host_agent/main.py` do in the
  applied build.

We redefine local copies of `AgentConnector` / `AgentDiscovery` in this notebook (rather than
importing from the MCP-phase folder) so the notebook is self-contained and runnable on its own.


## Setup — Imports

In [ ]:
# ============ IMPORTS ============
import asyncio
import json
import os
import threading
import time
import uuid

import httpx
import uvicorn

# --- A2A SDK (the same package already used in mcp_a2a_agentic_rag) ---
from a2a.types import (
    AgentCard,
    AgentSkill,
    AgentCapabilities,
    Message,
    MessageSendParams,
    Part,
    Role,
    SendMessageRequest,
    Task,
    TaskState,
    TextPart,
    UnsupportedOperationError,
)
from a2a.client import A2AClient, A2ACardResolver
from a2a.server.apps import A2AStarletteApplication
from a2a.server.request_handlers import DefaultRequestHandler
from a2a.server.tasks import InMemoryTaskStore, TaskUpdater
from a2a.server.agent_execution import AgentExecutor, RequestContext
from a2a.server.events import EventQueue
from a2a.utils import new_task, new_agent_text_message
from a2a.utils.errors import ServerError

# --- LangGraph (Research Agent only) ---
from langgraph.prebuilt import create_react_agent
from langchain_core.tools import tool

# --- Repo-wide LLM factory (never instantiate a provider client directly) ---
from helpers import get_llm

## Section 1 — Agent A: the "Research Agent" (LangGraph + a tool)

This agent is a small LangGraph ReAct agent built with `create_react_agent`. It has one
tool, `lookup_facts`, that stands in for a web-search / knowledge-lookup call. Its job is to
gather a handful of facts about whatever topic it is asked about — nothing more.


In [ ]:
# ============ RESEARCH AGENT: TOOL ============
_MOCK_KNOWLEDGE_BASE = {
    "solar power": [
        "Global solar capacity passed 1.6 terawatts by the mid-2020s.",
        "Solar panel costs have fallen more than 80% over the last decade.",
        "Perovskite-silicon tandem cells are pushing lab efficiencies past 33%.",
    ],
    "agent to agent protocol": [
        "A2A (Agent-to-Agent) is an open protocol for interoperability between AI agents.",
        "Agents advertise capabilities via a machine-readable AgentCard at /.well-known/agent.json.",
        "A2A is designed to be framework-agnostic: a LangGraph agent and a plain-LLM agent can both speak it.",
    ],
}

@tool
def lookup_facts(topic: str) -> str:
    """Look up a short list of known facts about a topic (stand-in for a web search tool)."""
    topic_key = topic.strip().lower()
    for key, facts in _MOCK_KNOWLEDGE_BASE.items():
        if key in topic_key or topic_key in key:
            return "\n".join(f"- {fact}" for fact in facts)
    return f"No indexed facts found for '{topic}'. Respond using general knowledge instead."


In [ ]:
# ============ RESEARCH AGENT: LANGGRAPH AGENT ============
research_llm = get_llm(temperature=0)

research_langgraph_agent = create_react_agent(
    model=research_llm,
    tools=[lookup_facts],
    prompt=(
        "You are a Research Agent. Given a topic, use the lookup_facts tool to gather "
        "factual bullet points, then return ONLY a concise, plain-text list of the facts "
        "you found. Do not add commentary, opinions, or a conclusion."
    ),
)


### Wrapping the Research Agent as an A2A Server

This follows the same `AgentExecutor` + `TaskUpdater` pattern as
`agents/agentic_rag_agent/agent_executor.py` in the applied build: pull the user input off
the `RequestContext`, invoke the underlying agent, and stream a `completed` status update
back with the final text.


In [ ]:
# ============ RESEARCH AGENT: A2A EXECUTOR ============
class ResearchAgentExecutor(AgentExecutor):
    """Wraps the LangGraph research agent so it can be served over A2A."""

    def __init__(self, langgraph_agent):
        self.agent = langgraph_agent

    async def execute(self, context: RequestContext, event_queue: EventQueue) -> None:
        query = context.get_user_input()
        task = context.current_task

        if not task:
            task = new_task(context.message)
            await event_queue.enqueue_event(task)

        updater = TaskUpdater(event_queue, task.id, task.context_id)

        try:
            result = await self.agent.ainvoke({"messages": [("user", query)]})
            final_text = result["messages"][-1].content

            await updater.update_status(
                TaskState.completed,
                new_agent_text_message(final_text, task.context_id, task.id),
            )
        except Exception as exc:
            await updater.update_status(
                TaskState.failed,
                new_agent_text_message(f"Research Agent error: {exc}", task.context_id, task.id),
            )
            raise

    async def cancel(self, request: RequestContext, event_queue: EventQueue) -> Task | None:
        raise ServerError(error=UnsupportedOperationError())


In [ ]:
# ============ RESEARCH AGENT: AGENT CARD ============
research_skill = AgentSkill(
    id="topic_research_skill",
    name="topic_research_skill",
    description="Gathers concise factual bullet points about a topic using a lookup tool.",
    tags=["research", "search", "facts"],
    examples=["Gather facts about solar power.", "Research the A2A protocol."],
)

RESEARCH_AGENT_HOST = "localhost"
RESEARCH_AGENT_PORT = 10101

research_agent_card = AgentCard(
    name="research_agent",
    description="A LangGraph ReAct agent that gathers facts about a topic using a lookup tool.",
    url=f"http://{RESEARCH_AGENT_HOST}:{RESEARCH_AGENT_PORT}/",
    version="1.0.0",
    defaultInputModes=["text"],
    defaultOutputModes=["text"],
    skills=[research_skill],
    capabilities=AgentCapabilities(streaming=True),
)


## Section 2 — Agent B: the "Writer Agent" (plain LLM call, no framework)

By design, this agent is structured nothing like the Research Agent. There is no
LangGraph, no tool loop, no agent framework at all — just a single `get_llm(...).invoke(...)`
call behind a fixed persona system prompt. A2A doesn't care: as long as it exposes a valid
`AgentCard` and speaks the same A2A wire protocol, the orchestrator can talk to it exactly
like it talks to the Research Agent.


In [ ]:
# ============ WRITER AGENT: PLAIN LLM PERSONA (no LangGraph, no tools) ============
writer_llm = get_llm(temperature=0.7)

WRITER_SYSTEM_PROMPT = (
    "You are a polished technical copywriter. You will be given raw research notes "
    "(a list of facts). Turn them into a short, engaging paragraph (3-5 sentences) "
    "suitable for a blog post intro. Do not simply list the facts back - synthesize them "
    "into flowing prose. Do not invent facts that were not given to you."
)

def write_from_notes(research_notes: str) -> str:
    """A single direct LLM call with a persona prompt - intentionally NOT an agent loop."""
    response = writer_llm.invoke(
        [
            ("system", WRITER_SYSTEM_PROMPT),
            ("user", research_notes),
        ]
    )
    return response.content


In [ ]:
# ============ WRITER AGENT: A2A EXECUTOR ============
class WriterAgentExecutor(AgentExecutor):
    """Wraps the plain-LLM writer function so it can be served over A2A."""

    def __init__(self, write_fn):
        self.write_fn = write_fn

    async def execute(self, context: RequestContext, event_queue: EventQueue) -> None:
        research_notes = context.get_user_input()
        task = context.current_task

        if not task:
            task = new_task(context.message)
            await event_queue.enqueue_event(task)

        updater = TaskUpdater(event_queue, task.id, task.context_id)

        try:
            final_text = self.write_fn(research_notes)
            await updater.update_status(
                TaskState.completed,
                new_agent_text_message(final_text, task.context_id, task.id),
            )
        except Exception as exc:
            await updater.update_status(
                TaskState.failed,
                new_agent_text_message(f"Writer Agent error: {exc}", task.context_id, task.id),
            )
            raise

    async def cancel(self, request: RequestContext, event_queue: EventQueue) -> Task | None:
        raise ServerError(error=UnsupportedOperationError())


In [ ]:
# ============ WRITER AGENT: AGENT CARD ============
writer_skill = AgentSkill(
    id="polished_writing_skill",
    name="polished_writing_skill",
    description="Turns raw research notes into a short, polished piece of prose.",
    tags=["writing", "content", "persona"],
    examples=["Turn these facts into a blog intro paragraph."],
)

WRITER_AGENT_HOST = "localhost"
WRITER_AGENT_PORT = 10102

writer_agent_card = AgentCard(
    name="writer_agent",
    description="A plain-LLM persona agent (no framework, no tools) that writes polished prose from notes.",
    url=f"http://{WRITER_AGENT_HOST}:{WRITER_AGENT_PORT}/",
    version="1.0.0",
    defaultInputModes=["text"],
    defaultOutputModes=["text"],
    skills=[writer_skill],
    capabilities=AgentCapabilities(streaming=True),
)


## Section 3 — Serving Both Agents over A2A

Each agent is built into an `A2AStarletteApplication` (same as `main.py` in the applied
build) and run with `uvicorn` in its own background thread, so both servers are live at
once inside this single notebook process.


In [ ]:
# ============ GENERIC A2A SERVER LAUNCHER ============
def build_a2a_app(agent_card: AgentCard, executor: AgentExecutor):
    request_handler = DefaultRequestHandler(
        agent_executor=executor,
        task_store=InMemoryTaskStore(),
    )
    server = A2AStarletteApplication(agent_card=agent_card, http_handler=request_handler)
    return server.build()


def run_server_in_background(app, host: str, port: int) -> uvicorn.Server:
    """Runs a uvicorn server on its own thread + event loop so the notebook stays free."""
    config = uvicorn.Config(app, host=host, port=port, log_level="warning")
    server = uvicorn.Server(config)

    def _run():
        asyncio.run(server.serve())

    thread = threading.Thread(target=_run, daemon=True)
    thread.start()
    return server


In [ ]:
# ============ LAUNCH BOTH AGENT SERVERS ============
research_app = build_a2a_app(research_agent_card, ResearchAgentExecutor(research_langgraph_agent))
writer_app = build_a2a_app(writer_agent_card, WriterAgentExecutor(write_from_notes))

research_server = run_server_in_background(research_app, RESEARCH_AGENT_HOST, RESEARCH_AGENT_PORT)
writer_server = run_server_in_background(writer_app, WRITER_AGENT_HOST, WRITER_AGENT_PORT)

time.sleep(1.5)  # give both uvicorn servers a moment to bind their ports

print(f"Research Agent serving at {research_agent_card.url}")
print(f"Writer Agent serving at   {writer_agent_card.url}")


## Section 4 — Agent Registry + Discovery

Mirroring `agent_discovery.py`, we write a small registry file listing both agents' base
URLs, then use an `AgentDiscovery`-style class to fetch each one's `AgentCard` from its
`/.well-known/agent.json` endpoint. Neither agent's internals are inspected here - only
the standard A2A discovery document.


In [ ]:
# ============ WRITE THE AGENT REGISTRY FILE ============
REGISTRY_PATH = os.path.join(os.getcwd(), "agent_registry.json")

with open(REGISTRY_PATH, "w") as f:
    json.dump(
        [research_agent_card.url, writer_agent_card.url],
        f,
        indent=2,
    )

print(f"Wrote agent registry to {REGISTRY_PATH}")


In [ ]:
# ============ AGENT DISCOVERY (mirrors utilities/a2a/agent_discovery.py) ============
class AgentDiscovery:
    """
    Discovers A2A Agents by reading a registry file of URLs and
    querying each one's /.well-known/agent.json endpoint to retrieve
    an AgentCard.
    """

    def __init__(self, registry_file: str = None):
        self.registry_file = registry_file or os.path.join(os.getcwd(), "agent_registry.json")
        self.base_urls = self._load_registry()

    def _load_registry(self) -> list[str]:
        try:
            with open(self.registry_file, "r") as f:
                data = json.load(f)
            if not isinstance(data, list):
                raise ValueError("Registry file must contain a list of URLs.")
            return data
        except FileNotFoundError:
            print(f"Registry file '{self.registry_file}' not found.")
            return []
        except (json.JSONDecodeError, ValueError) as e:
            print(f"Error parsing registry file: {e}")
            return []

    async def list_agent_cards(self) -> list[AgentCard]:
        cards: list[AgentCard] = []
        async with httpx.AsyncClient(timeout=300.0) as httpx_client:
            for base_url in self.base_urls:
                resolver = A2ACardResolver(base_url=base_url.rstrip("/"), httpx_client=httpx_client)
                card = await resolver.get_agent_card()
                cards.append(card)
        return cards


In [ ]:
# ============ DISCOVER BOTH AGENTS ============
discovery = AgentDiscovery(registry_file=REGISTRY_PATH)
discovered_cards = await discovery.list_agent_cards()

for card in discovered_cards:
    print(f"Discovered agent: {card.name!r} -> {card.url}")
    print(f"  description: {card.description}")


## Section 5 — Orchestration: Research Agent -> Writer Agent, purely over A2A

The orchestrator here is deliberately dumb: it knows nothing about LangGraph or about the
Writer Agent's persona prompt. It only knows how to talk `AgentCard` + `AgentConnector.send_task`.
This is the actual value proposition of A2A - the orchestrator (and each agent) only needs
to know the *protocol*, never the other side's implementation.


In [ ]:
# ============ AGENT CONNECTOR (mirrors utilities/a2a/agent_connect.py, updated to the current a2a-sdk API) ============
# NOTE: the existing utilities/a2a/agent_connect.py in mcp_a2a_agentic_rag calls a
# `A2AClient(base_url=...)` / `client.send_request(...)` API that no longer exists in
# current a2a-sdk releases. This connector uses the verified-current API instead
# (A2AClient(httpx_client=, agent_card=) + client.send_message(SendMessageRequest(...))),
# the same pattern demonstrated in 01_Foundations/01_A2A_Protocol_Basics.ipynb.
class AgentConnector:
    """Simple A2A Agent Connector that facilitates communication with other agents."""

    def __init__(self, agent_card: AgentCard):
        self.agent_card = agent_card

    async def send_task(self, message: str, session_id: str) -> str:
        try:
            async with httpx.AsyncClient(timeout=300.0) as httpx_client:
                client = A2AClient(httpx_client=httpx_client, agent_card=self.agent_card)

                user_message = Message(
                    role=Role.user,
                    parts=[Part(root=TextPart(text=message))],
                    message_id=str(uuid.uuid4()),
                )
                request = SendMessageRequest(
                    id=str(uuid.uuid4()),
                    params=MessageSendParams(message=user_message),
                )
                response = await client.send_message(request)

                result = response.root.result
                if hasattr(result, "parts"):  # a Message
                    return "\n".join(
                        part.root.text for part in result.parts if hasattr(part.root, "text")
                    )
                return str(result)  # a Task object; fall back to its string representation
        except Exception as e:
            return f"Error communicating with agent: {str(e)}"


In [ ]:
# ============ ORCHESTRATE: RESEARCH -> WRITE, HOP BY HOP ============
research_card = next(c for c in discovered_cards if c.name == "research_agent")
writer_card = next(c for c in discovered_cards if c.name == "writer_agent")

research_connector = AgentConnector(research_card)
writer_connector = AgentConnector(writer_card)

session_id = "cross-framework-demo-session"
topic = "the agent to agent protocol"

print(f"[HOP 1] Orchestrator -> Research Agent ({research_card.url})")
print(f"        sending topic: {topic!r}")

research_output = await research_connector.send_task(
    message=f"Gather facts about: {topic}",
    session_id=session_id,
)

print(f"[HOP 1] Research Agent returned:\n{research_output}\n")

print(f"[HOP 2] Orchestrator -> Writer Agent ({writer_card.url})")
print("        forwarding the Research Agent's raw output as input, unmodified")

final_writeup = await writer_connector.send_task(
    message=research_output,
    session_id=session_id,
)

print(f"[HOP 2] Writer Agent returned:\n{final_writeup}")


In [ ]:
# ============ FINAL RESULT ============
print("=" * 60)
print("FINAL POLISHED OUTPUT (produced entirely via A2A handoff)")
print("=" * 60)
print(final_writeup)


### Discussion of the Output

Walking back through what just happened:

- The orchestrator never imported anything from either agent's implementation. It only
  ever called `AgentConnector.send_task(...)` against an `AgentCard` it discovered from a
  `/.well-known/agent.json` document.
- The **Research Agent** is a full LangGraph ReAct loop with a tool-calling step; the
  **Writer Agent** is one `get_llm().invoke(...)` call with a system prompt. Structurally
  they have almost nothing in common.
- Yet the handoff between them worked purely through the A2A message contract: `HOP 1`
  shows the Research Agent's raw factual bullets going out over A2A, and `HOP 2` shows
  those same bullets going *back in* to a completely unrelated agent, which returns
  polished prose.
- This is the actual value proposition of A2A: it lets you swap, upgrade, or completely
  reimplement either agent's internals (different framework, different model, different
  prompting strategy) without touching the orchestrator or the other agent at all, as long
  as the `AgentCard` contract stays the same.


## Relationship to `mcp_a2a_agentic_rag`

This notebook's `ResearchAgentExecutor` / `WriterAgentExecutor` + `AgentConnector` +
`AgentDiscovery` classes are a deliberately minimal, RAG-free, MCP-free version of the same
pattern used in `09_Agent_Protocols/MCP/mcp_a2a_agentic_rag/`:

- That applied build's `agents/agentic_rag_agent/` plays a role analogous to our Research
  Agent here (an agent with real document/tool-backed capability, wrapped in an
  `AgentExecutor` and served via `A2AStarletteApplication`).
- Its `agents/host_agent/` plays a role analogous to our orchestrator: it discovers other
  agents via `AgentDiscovery` and dispatches tasks to them via `AgentConnector`, except at
  larger scale it *also* pulls in MCP tools alongside the A2A agent calls.

The `mcp_a2a_agentic_rag` build layers RAG and MCP tool access on top of the same A2A
skeleton demonstrated here. If you understood this notebook's Section 3-5, you already
understand the A2A backbone of that larger system - the rest is RAG plumbing and MCP tool
wiring bolted onto agents that speak A2A exactly like `research_agent` and `writer_agent`
do here.


## Summary — Key Takeaways

- **A2A's core value is protocol-level interoperability, not framework unification.** Two
  agents with completely different internals (a LangGraph ReAct agent with tools vs. a
  single plain-LLM persona call) collaborated with zero shared code, only a shared
  `AgentCard` + task-message contract.
- **Discovery is registry-driven and standards-based.** `AgentDiscovery` reads a plain list
  of base URLs and resolves each agent's capabilities from its own
  `/.well-known/agent.json`, so agents can be added, removed, or reimplemented without
  changing the orchestrator's code.
- **`AgentConnector` is the only thing the orchestrator needs.** A single
  `send_task(message, session_id)` call is enough to hand work to any A2A-compliant agent,
  regardless of what's running behind its `AgentCard`.
- **This is the same backbone used at production scale.** `09_Agent_Protocols/MCP/mcp_a2a_agentic_rag/`
  applies this exact `AgentCard` / `AgentExecutor` / `AgentConnector` / `AgentDiscovery`
  pattern to coordinate a real agentic-RAG agent and a host agent, with MCP tools layered
  on top.

This notebook is the **capstone of a 3-part A2A track**:

1. `01_Foundations/01_A2A_Protocol_Basics.ipynb` - the protocol itself: `AgentCard`,
   `AgentSkill`, tasks, and the `/.well-known/agent.json` discovery document.
2. `02_Building_Agents_with_A2A/01_LangGraph_Agent_over_A2A.ipynb` - wrapping a single
   LangGraph agent as an A2A server.
3. **This notebook** - two differently-built agents interoperating purely through A2A,
   with a discovery registry and a hop-by-hop orchestrated handoff between them.
